In [1]:
!python -m pip install git+https://github.com/openradar/xradar.git

  Cloning https://github.com/openradar/xradar.git to /tmp/pip-req-build-jjnr0jze
  Running command git clone --filter=blob:none --quiet https://github.com/openradar/xradar.git /tmp/pip-req-build-jjnr0jze
  Resolved https://github.com/openradar/xradar.git to commit 059b5bf047243c06cfcd4a567c8e2a126da256ab
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import fsspec
import xradar as xd
import fsspec
import numcodecs
import os
import time
from xarray import DataTree

In [3]:

# Download and open your NEXRAD Level 2 file
file = "s3://noaa-nexrad-level2/2022/03/22/KHGX/KHGX20220322_120125_V06"
local_file = fsspec.open_local(f'simplecache::{file}', s3={'anon': True}, filecache={'cache_storage': '.'})
dtree = xd.io.open_nexradlevel2_datatree(local_file)

In [12]:
def dtree_encoding(dtree, append_dim, compressor) -> dict:
    """
    Generate encoding for each variable in the DataTree with scale and offset for numeric variables.
    
    Parameters:
        dtree (DataTree): Input xarray DataTree.
        append_dim (str): Dimension to encode (e.g., "vcp_time").
        compressor (numcodecs.abc.Codec): Compression codec (e.g., numcodecs.Blosc).
    
    Returns:
        dict: Encoding dictionary for use in to_zarr().
    """
    _encoding = {}
    for node in dtree.match("*/sweep_*").subtree:
        if node.path == "/":
            continue
        if node.path not in _encoding:
            _encoding[node.path] = {}
        
        for var_name, var_data in node.to_dataset().data_vars.items():
            if var_data.dtype.kind == "f":  # Only apply scale/offset to float variables
                _encoding[node.path][var_name] = {
                    "_FillValue": -9999,
                    "compressor": compressor,
                    "dtype": "float32",  # Keep dtype float32 to avoid casting errors
                    "scale_factor": 10,
                    "add_offset": -33
                }
            elif var_data.dtype.kind in {"i"}:  # For integer variables
                _encoding[node.path][var_name] = {
                    "compressor": compressor,
                    "dtype": var_data.dtype
                }
            else:
                # For non-numeric variables, just apply compression
                _encoding[node.path][var_name] = {"compressor": compressor}

        _encoding.pop("/", None)

    return _encoding


In [16]:
def benchmark_dtree_compression(dtree, compressors, append_dim, local_file):
    """
    Benchmark saving a DataTree with different compressors and measure file size and time.
    
    Parameters:
        dtree (DataTree): Input xarray DataTree.
        compressors (list): List of numcodecs compressors to test.
        append_dim (str): Dimension to encode (e.g., "vcp_time").
        local_file (str): Path to the original local file for size comparison.
    
    Returns:
        list of dict: Benchmark results with file size and time for each compressor and the original file.
    """
    results = []

    # Get size of the original file in bytes and convert to MB
    original_file_size_bytes = os.path.getsize(local_file)
    original_file_size_mb = original_file_size_bytes / (1024 * 1024)

    # Add a row for the original file size (without compression)
    results.append({
        "Compressor": "Original File",
        "File Size (Bytes)": original_file_size_bytes,
        "File Size (MB)": round(original_file_size_mb, 2),
        "Time (s)": "N/A"  # No time measurement for the original file
    })

    for compressor in compressors:
        compressor_name = compressor.codec_id if hasattr(compressor, "codec_id") else str(compressor)
        print(f"Testing compressor: {compressor_name}")

        # Create encoding with the specified compressor
        encoding = dtree_encoding(dtree, append_dim, compressor)

        # Measure time and file size
        file_path = f"benchmark_{compressor_name}_{}.zarr"
        start_time = time.time()
        dtree.to_zarr(file_path, mode="w", encoding=encoding)
        elapsed_time = time.time() - start_time
        
        # Measure the size of the compressed file in bytes
        compressed_file_size_bytes = sum(
            os.path.getsize(os.path.join(dirpath, f))
            for dirpath, _, files in os.walk(file_path)
            for f in files
        )
        compressed_file_size_mb = compressed_file_size_bytes / (1024 * 1024)

        results.append({
            "Compressor": compressor_name,
            "File Size (Bytes)": compressed_file_size_bytes,
            "File Size (MB)": round(compressed_file_size_mb, 2),
            "Time (s)": round(elapsed_time, 2)
        })
        
        print(f"Finished {compressor_name}: {compressed_file_size_mb:.2f} MB in {elapsed_time:.2f}s\n")
    
    return results



In [ ]:

# List of compressors to test
compressors = [
    numcodecs.Zlib(level=1),
    numcodecs.Zlib(level=5),
    numcodecs.Zlib(level=9),
    numcodecs.Blosc(cname="zstd", clevel=1, shuffle=numcodecs.Blosc.SHUFFLE),
    numcodecs.Blosc(cname="zstd", clevel=5, shuffle=numcodecs.Blosc.SHUFFLE),
    numcodecs.Blosc(cname="zstd", clevel=9, shuffle=numcodecs.Blosc.SHUFFLE),
    numcodecs.Blosc(cname="lz4", clevel=1, shuffle=numcodecs.Blosc.SHUFFLE),
    numcodecs.Blosc(cname="lz4", clevel=5, shuffle=numcodecs.Blosc.SHUFFLE),
    numcodecs.Blosc(cname="lz4", clevel=9, shuffle=numcodecs.Blosc.SHUFFLE),
    # numcodecs.Blosc(cname="snappy", clevel=1, shuffle=numcodecs.Blosc.SHUFFLE),
    # numcodecs.Blosc(cname="snappy", clevel=5, shuffle=numcodecs.Blosc.SHUFFLE),
]

# Run the benchmark
results = benchmark_dtree_compression(dtree, compressors, append_dim="vcp_time", local_file=local_file)

# Display results
import pandas as pd
df = pd.DataFrame(results)
print(df)


Testing compressor: zlib
Finished zlib: 32.42 MB in 2.62s

Testing compressor: zlib
Finished zlib: 27.39 MB in 5.29s

Testing compressor: zlib
Finished zlib: 26.04 MB in 34.05s

Testing compressor: blosc
Finished blosc: 44.59 MB in 1.47s

Testing compressor: blosc
Finished blosc: 40.04 MB in 4.93s

Testing compressor: blosc


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


# Convert to DataFrame
df_results = df

# Plot: File Size Comparison
plt.figure(figsize=(12, 6))
plt.bar(df_results["Compressor"], df_results["File Size (MB)"])
plt.xticks(rotation=45, ha="right")
plt.title("File Size Comparison (MB)")
plt.xlabel("Compressor")
plt.ylabel("File Size (MB)")
plt.tight_layout()
plt.show()

# Plot: Time Comparison (ignoring Original File as it has no time measurement)
df_results_time = df_results[df_results["Time (s)"].notnull()]

plt.figure(figsize=(12, 6))
plt.bar(df_results_time["Compressor"], df_results_time["Time (s)"])
plt.xticks(rotation=45, ha="right")
plt.title("Compression Time Comparison (s)")
plt.xlabel("Compressor")
plt.ylabel("Time (s)")
plt.tight_layout()
plt.show()
